# Axion · Etapa 2: instrumento, línea base y continuidad

**Córrelo solo después de que `Axion_fase0b_v5` haya terminado bien y Claude haya revisado su diagnóstico.**

Pasos: GPU T4 → *Entorno de ejecución → Ejecutar todo* → pegar la clave de Gemini cuando aparezca el cuadro → esperar **LISTO** → avisarle a Claude.

Todo queda en Drive, carpeta `Axion_v4`, en `diagnostico_etapa2.txt`.

## Ajustes
Cada bloque caro se prende por separado. La primera vez deja todo como está: compara las anclas, crea el instrumento, aplica la línea base de palabras y mide cuánto tarda Gemini en responder. Son pocos minutos y casi no gasta llamadas.

In [ ]:
VARIANTE_ANCLAS = 'parafraseado'        # 'una_linea' o 'parafraseado' (se decide mirando la comparacion)
CORRER_CONTINUIDAD = False              # experimento con y sin historial (≈60 llamadas)
REPETICIONES_CONTINUIDAD = 2            # subir a 5 o mas cuando funcione
CORRER_REPLICAS = False                 # 7 sesiones x N corridas (≈105 llamadas por corrida)
N_REPLICAS = 3

MODELOS_SUJETO = ['gemini-3.8-flash', 'gemini-3.6-flash', 'gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']
MODELOS_OBSERVADOR = ['gemini-3.6-flash', 'gemini-3.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-3.8-flash']
CARPETA = '/content/drive/MyDrive/Axion_v4'

In [ ]:
!pip -q install sentence-transformers google-genai scipy

## 0 · Preparación

In [ ]:
import os, sys, json, glob, platform
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir(CARPETA), 'No encuentro la carpeta Axion_v4 en tu Drive'
os.chdir(CARPETA); sys.path.insert(0, CARPETA)
from colab_util import C, ESTADO, DIAG, iniciar, etapa, pedir_clave, elegir_modelo, cierre
iniciar('registro_etapa2.txt', 'diagnostico_etapa2.txt')

@etapa('0 entorno')
def _():
    import torch, transformers, sentence_transformers
    necesarios = ('manifold_v5.py', 'fase0b_real_v2.py', 'meditacion_ncl_v34.py', 'instrumento_axion.py',
                  'analisis_extra.py', 'experimento_continuidad.py')
    faltan = [a for a in necesarios if not os.path.exists(a)]
    assert not faltan, f'Faltan archivos en Axion_v4: {faltan}'
    DIAG['entorno'] = {'python': platform.python_version(), 'torch': torch.__version__,
                       'transformers': transformers.__version__, 'sentence_transformers': sentence_transformers.__version__,
                       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU'}
    print(DIAG['entorno'])

## 1 · Comparar variantes de anclas y congelar el instrumento
- `discriminacion` (azar 0,50) e `identificacion` (azar 0,09): ¿el eje reconoce su polo dicho con otras palabras?
- `solape_entre_ejes`: mientras más bajo, más distintos son los ejes entre sí.
- El instrumento elegido se guarda en un archivo `instrumento_<id>.json`. Si ya existe uno con ese id, se reutiliza tal cual.

In [ ]:
@etapa('1 instrumento', requiere=['0 entorno'])
def _():
    from manifold_v5 import HarrierEmbedder
    from instrumento_axion import comparar_variantes, crear_instrumento, ManifoldInstrumento
    emb = HarrierEmbedder()
    DIAG['comparacion_variantes'] = comparar_variantes(emb)
    for v, r in DIAG['comparacion_variantes'].items():
        print(v, {k: x for k, x in r.items() if 'por_ancla' not in k})
    m = crear_instrumento(emb, VARIANTE_ANCLAS)
    ruta = f'instrumento_{m.instrumento_id}.json'
    if os.path.exists(ruta):
        m = ManifoldInstrumento.cargar(ruta, emb); print('Instrumento existente reutilizado:', ruta)
    else:
        m.guardar(ruta); print('Instrumento nuevo guardado:', ruta)
    DIAG['instrumento'] = {'id': m.instrumento_id, 'archivo': ruta, 'embedder': m.embedder_nombre}
    C['emb'], C['m'] = emb, m

## 2 · Línea base de palabras clave (no gasta llamadas)
Usa los resultados que ya existan de la etapa 1. Si la geometría no le gana a contar palabras, el manifold no está aportando.

In [ ]:
@etapa('2 linea base de palabras', requiere=['0 entorno'])
def _():
    from analisis_extra import comparar_con_palabras
    rutas = [r for r in ('resultados_fase0b_v2.json', 'prueba_sesion6_v2.json') if os.path.exists(r)]
    rutas += sorted(glob.glob('resultados_fase0b_*_r*.json'))
    assert rutas, 'Todavia no hay archivos de resultados para analizar.'
    DIAG['linea_base_palabras'] = {r: comparar_con_palabras(r) for r in rutas}
    print(json.dumps(DIAG['linea_base_palabras'], indent=1, ensure_ascii=False))

## 3 · Clave y modelos de Gemini

In [ ]:
pedir_clave()

@etapa('3 gemini', requiere=['0 entorno'])
def _():
    assert os.environ.get('GEMINI_API_KEY'), 'No hay clave de Gemini: vuelve a ejecutar esta celda y pegala en el cuadro.'
    C['sujeto'] = elegir_modelo(MODELOS_SUJETO)
    C['observador'] = elegir_modelo(MODELOS_OBSERVADOR, evitar=C['sujeto'].model)
    DIAG['modelos_usados'] = {'sujeto': C['sujeto'].model, 'observador': C['observador'].model}
    print(DIAG['modelos_usados'])

## 4 · Experimento de continuidad
¿El mismo modelo responde distinto cuando recibe el registro de su propia conversación anterior? Se compara contra recibir un registro que no viene al caso y contra no recibir nada.

In [ ]:
if CORRER_CONTINUIDAD:
    @etapa('4 continuidad', requiere=['1 instrumento', '3 gemini'])
    def _():
        from experimento_continuidad import ExperimentoContinuidad
        exp = ExperimentoContinuidad(C['m'], C['sujeto'], C['observador'], repeticiones=REPETICIONES_CONTINUIDAD)
        out = exp.run(f"resultados_continuidad_{C['m'].instrumento_id}.json")
        DIAG['continuidad'] = {'meta': out['meta'], 'analisis': out['analisis'],
                               'registro_propio': out['vivencia_propia']['registro'],
                               'registro_ajeno': out['vivencia_ajena']['registro'],
                               'ejemplos': [{k: r[k] for k in ('pregunta', 'condicion', 'respuesta', 'juez')}
                                            for r in out['respuestas'] if r['rep'] == 1]}
        print(json.dumps(out['analisis'], indent=1, ensure_ascii=False))
else:
    ESTADO['4 continuidad'] = 'no pedida'

## 5 · Réplicas de las 7 sesiones (solo si `CORRER_REPLICAS = True`)

In [ ]:
if CORRER_REPLICAS:
    @etapa('5 replicas', requiere=['1 instrumento', '3 gemini'])
    def _():
        from fase0b_real_v2 import Fase0bReal
        from analisis_extra import correr_replicas, resumen_replicas, comparar_con_palabras
        runner = Fase0bReal(C['m'], C['sujeto'], C['observador'])
        rutas = correr_replicas(runner, N_REPLICAS)
        DIAG['replicas'] = resumen_replicas(rutas)
        DIAG['replicas_palabras'] = {r: comparar_con_palabras(r) for r in rutas}
        print(json.dumps(DIAG['replicas'], indent=1, ensure_ascii=False))
else:
    ESTADO['5 replicas'] = 'no pedida'

## 6 · Cierre

In [ ]:
cierre(drive)